# MC2fit Tutorial: 1. Setting and Downloading

This notebook prepares Legacy Survey cutouts for **MC2fit**, a multi-component GALFIT-based pipeline from the [`galaxydiver/MC2fit`](https://github.com/galaxydiver/MC2fit) repository.

The goal of this notebook is **not yet to perform the final science fit**. Instead, it prepares the files that the fitting notebook will need later.


The explanations are written for a first-time user. Each major section describes:

- **Purpose** — why the step exists,
- **Inputs** — what files or variables are required,
- **Outputs** — what files or objects are created,
- **Common pitfalls** — things that can silently break the workflow.

## Package and module overview

This notebook combines standard astronomy tools with custom modules from the MC2fit repository.

### Standard Python / astronomy packages

| Package | Used for |
|---|---|
| `numpy`, `pandas` | arrays, tables, and target catalogs |
| `matplotlib` | visual inspection plots |
| `astropy.io.fits` | reading and writing FITS images |
| `astropy.wcs`, `Cutout2D` | astronomical image metadata and cutouts |
| `sep` | SExtractor-like source detection and segmentation |
| `scipy`, `opencv/cv2` | fitting, connected-component labeling, and image operations |

### MC2fit helper modules

| Module | Main role in this notebook |
|---|---|
| `DH_LegacyCrawling.py` | download Legacy Survey image products and PSF files |
| `DH_FileCheck.py` | create target directories, inspect downloaded files, generate links, and remove temporary files |
| `DH_Galfit_Sequence.py` | higher-level wrappers for image/PSF preparation |
| `DH_MC2fit.py` | define MC2fit/GALFIT settings and run configurations |
| `DH_masking.py`, `DH_masking2.py` | mask-combination and image-masking utilities |
| `DH_path.py` | point to the directory where summary files exist (change this) |

**Practical note:** this notebook assumes that the MC2fit repository is already available in the Python path. If imports fail, first check that you are running the notebook from an environment where these custom modules can be imported.

### Import cell

Run this cell first. If it fails, the most likely issue is that the MC2fit repository is not on your Python path.

A clean setup should import all modules without modifying `sys.path` inside the notebook. If you need to add a path manually, it is better to do it explicitly in a separate environment-setup cell.

In [1]:
# ============================================================
# Imports
# ============================================================
# Standard numerical and table-handling packages
import numpy as np
import pandas as pd

# Plotting and visualization
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.ticker import AutoMinorLocator

# General utilities
import pickle
import time
import copy
import random
import os
import glob
import shutil
from pathlib import Path
from dataclasses import dataclass
from multiprocessing import Pool
import subprocess
import warnings

# Astropy tools for statistics, FITS I/O, WCS, cutouts, and visualization
from astropy.stats import sigma_clip, sigma_clipped_stats
from astropy.io import fits
from astropy.nddata import Cutout2D
from astropy.utils.data import download_file
from astropy.wcs import WCS
from astropy.visualization import ZScaleInterval, ImageNormalize, AsinhStretch
from astropy.modeling.functional_models import Sersic1D, Sersic2D

# Image processing
import skimage
from skimage.transform import resize

# ============================================================
# Custom MC2fit modules
# Repository: https://github.com/galaxydiver/MC2fit
# ============================================================
# Array/path utilities
import DH_array as dharray
import DH_path as dhpath

# Download Legacy Survey images, PSF files, and related products
import DH_LegacyCrawling as dhcrawl

# File and directory management utilities
import DH_FileCheck as dhfc

# Main MC2fit configuration and execution tools
import DH_MC2fit as dhmc2fit


This machine : SO desktop computer
LegacyCrawling - Loading summary fits south :  /home/donghyeon/ua/udg/galfit/galfit/RawData/survey-bricks-dr9-south.fits
LegacyCrawling - Loading summary fits south (DR10) :  /home/donghyeon/ua/udg/galfit/galfit/RawData/survey-bricks-dr10-south.fits
LegacyCrawling - Loading summary fits north :  /home/donghyeon/ua/udg/galfit/galfit/RawData/survey-bricks-dr9-north.fits


### Working directory

The notebook changes into a project-specific working directory. This keeps downloaded files and generated masks separate from the source notebook.

**Pitfall:** because `os.chdir()` changes the active directory, rerunning this cell multiple times from a different starting location can cause path confusion. If that happens, restart the kernel and run from the original notebook directory.

In [2]:
## Create a dedicated working directory for this example and move into it.
dir_main = './Galfit_Run/PSB_new'

In [3]:
os.makedirs(dir_main, exist_ok=True)
os.chdir(dir_main)

# Initial setup

This section creates the working directory structure and reads the target catalog.

## Purpose

Before downloading images or running masks, each target needs a consistent directory. MC2fit expects a predictable folder layout so that image, PSF, sigma, and mask files can be found automatically.

## Required input

- `catalog`
- columns named `RA` and `dec`
- optionally a target-name column 

## Main outputs

- `DIdat`: directory manager for raw downloaded survey products
- `coord_array`: target coordinates used by the download functions

## Setup the paremeter, load the catalog

In [4]:
# Number of CPU workers to use for multicore tasks.
# Use a moderate value for network downloads. Too many simultaneous requests
# can trigger server-side rate limits or timeout errors.
Ncore = 4

# Legacy Survey pixel scale in arcsec / pixel.
# This is needed when converting pixel-based sizes to angular units.
plate_scale = 0.262

# Cutout size in pixels.
# For image_size=121, each cutout is roughly 121 * 0.262 arcsec across.
image_size = 121

## Estimated sky value 
est_sky=10 

# Load the target catalog.
# The catalog must contain at least RA and dec columns.
fn = "./fulldata.csv"
target_catalog = pd.read_csv(fn)
target_catalog


,name,classification,discovery_date,ra,dec,transient_redshift,host_name,host_skycoord,host_redshift
0,WTP14adbwvs,TDE,NaN,198.748944,51.272774,0.02489,Nan_named,NaN,NaN
1,WTP17aaldjb,TDE,NaN,197.064964,4.485906,0.04832,Nan_named,NaN,NaN
2,2020nov,TDE,59027.232558,254.554042,2.117511,0.08400,Nan_named,NaN,NaN
3,2021axu,TDE,59231.512000,176.651535,30.085454,0.19000,Nan_named,NaN,NaN
4,2020abri,TDE,59184.529051,202.321980,19.670974,0.17800,Nan_named,NaN,NaN
...,...,...,...,...,...,...,...,...,...
233,RX J1301.9+2747,QPE,58633.000000,195.500575,27.782736,0.02400,NaN,NaN,NaN
234,eR0-QPE1,QPE,59057.805000,37.946960,-10.336197,0.05000,NaN,NaN,NaN
235,eR0-QPE2,QPE,59067.846000,38.703000,-44.325690,0.01800,NaN,NaN,NaN
236,eRO-QPE3,QPE,58874.111000,210.222200,-28.766500,0.02400,NaN,NaN,NaN


In [5]:
# Extract sky coordinates used by the Legacy Survey download functions.
# The coordinate array has shape (N_target, 2), with columns [RA, dec].
col = ['ra', 'dec']
coord_array = target_catalog[col].to_numpy().astype(float)

# Quick sanity check: display the RA/Dec array that will be passed to download functions.
len(coord_array)

238

## Target catalog and directory managers

`DH_FileCheck.DirInfo` is a bookkeeping object. It does not perform the science analysis itself; instead, it stores:

- where each target lives,
- how target folders are named,
- the coordinate list,
- target IDs,
- optional catalog names.

In this notebook we use two related directory trees:

- `gal_dat/`: raw downloaded products from Legacy Survey, (See, `DIdat`)
- `gal_out/`: cleaned/standardized products used later by MC2fit/GALFIT. (See, `DI`)

Keeping these separate is useful because raw downloads may include north/south alternatives, failed files, or temporary error logs, while the fitting directory should contain only the final chosen files.

In [6]:
# DI manages the standardized fitting/output directory tree.
# Each target gets a folder named with folder_rule, e.g. gal0000/, gal0001/, ...
DIdat = dhfc.DirInfo(
    './',
    dir_work_base='gal_dat/',          # downloaded files
    folder_rule="gal%04d/",            # four-digit folder names; supports up to 9999 targets. To use five-digit names, use "gal%05d/"
    coord_array=coord_array,
    idlist=np.arange(len(coord_array)), # internal target IDs
    catalogname_list=target_catalog['name'],    # human-readable target names
)

# Print the number of target directories that will be created (= number of targets)
print(len(DIdat.dir_work_list))


238


In [7]:
# Actually create the output folders defined by the DirInfo objects.
# These methods only create directories; they do not download or process data.
DIdat.make_dir_work()

# Download Legacy Survey products

This section downloads the astronomical images and PSF models used for fitting.

## Why downloading is split into multiple steps

Legacy Survey requests sometimes fail because of:

- temporary server timeouts,
- too many parallel requests,
- network interruptions,
- missing products for a specific target or survey region.

For that reason, this notebook first downloads PSF files, checks which targets succeeded, then downloads main image products and performs quality checks.

## 2.1 Download PSF models

The **PSF** describes how a point source is blurred by the telescope, detector, atmosphere, and image processing. GALFIT/MC2fit needs the PSF to compare model galaxies to the observed image fairly.

For each target, this step attempts to download PSF model FITS files from the Legacy Survey database.

Typical outputs include files such as:

- `psf_model_dr9_s.fits`
- `psf_model_dr9_n.fits`
- `psf_model_dr10_s.fits`

where `n` and `s` refer to north/south survey regions.

### PSF download strategy

The first download pass is intentionally conservative:

- fewer cores,
- short timeout,
- minimal retry.

This quickly retrieves files that are easy to access. Later cells retry missing files with longer timeouts. This avoids spending too much time waiting on problematic targets during the first pass.

In [13]:
## First pass: download only the PSF models.
## In DH_LegacyCrawling.py, multicore_download_cutout() queries Legacy Survey PSF endpoints.
results = dhcrawl.multicore_download_cutout(
    DIdat,
    Ncore=3,
    overwrite=False, ## Re-download all files
    Ntry=1,
    timeout=5,
    silent_error=True,  ## Skip the error message
    silent=True ## Hide detailed info
    
)


Start Multicore run
======== Parallel Briefing ========
● Function: partial
● Number of loops: 238
● Number of workers: 3
● Backend: thread
● Using try/except: False


partial [thread]:   0%|          | 0/238 [00:00<?, ?it/s]

>> Time out error! ./gal_dat/gal0011/psf_model_dr9_s.fits
>> Time out error! ./gal_dat/gal0045/psf_model_dr10_s.fits
>> Time out error! ./gal_dat/gal0034/psf_model_dr10_s.fits
>> Time out error! ./gal_dat/gal0058/psf_model_dr9_s.fits
>> Time out error! ./gal_dat/gal0061/psf_model_dr9_s.fits
>> Time out error! ./gal_dat/gal0088/psf_model_dr10_s.fits
>> Time out error! ./gal_dat/gal0103/psf_model_dr9_s.fits
>> Time out error! ./gal_dat/gal0093/psf_model_dr10_s.fits
>> Time out error! ./gal_dat/gal0126/psf_model_dr9_s.fits
>> Time out error! ./gal_dat/gal0103/psf_model_dr10_s.fits
>> Time out error! ./gal_dat/gal0133/psf_model_dr10_s.fits
>> Time out error! ./gal_dat/gal0126/psf_model_dr10_s.fits
>> Time out error! ./gal_dat/gal0146/psf_model_dr9_s.fits
>> Time out error! ./gal_dat/gal0153/psf_model_dr9_s.fits
>> Time out error! ./gal_dat/gal0146/psf_model_dr10_s.fits
>> Time out error! ./gal_dat/gal0158/psf_model_dr10_s.fits
>> Time out error! ./gal_dat/gal0153/psf_model_dr10_s.fits
>> T

### Checking PSF completeness

The `FileGroupSearch` object checks whether each target has the requested file set. For PSFs, we are usually satisfied if at least one appropriate survey-region PSF exists.

The diagnostic expression below identifies targets that are missing both north and south PSF products.

If many targets are missing, reduce `Ncore`, increase `timeout`, or retry later.

In [10]:
## Check which files now exist.
FG = dhfc.FileGroupSearch(
    DirInfo=DIdat,
    fn_set=['psf_model_dr9_s.fits', 'psf_model_dr9_n.fits', 'psf_model_dr10_s.fits']
)

Gal Index:  0 - 237
Total NGal :  238
>> File: psf_model_dr9_s.fits  -  12 / 238
>> Err file: err_psf_model_dr9_s.dat  -  72 / 238
>> Total: psf_model_dr9_s.fits  -  84 / 238


>> File: psf_model_dr9_n.fits  -  46 / 238
>> Err file: err_psf_model_dr9_n.dat  -  116 / 238
>> Total: psf_model_dr9_n.fits  -  162 / 238


>> File: psf_model_dr10_s.fits  -  23 / 238
>> Err file: err_psf_model_dr10_s.dat  -  58 / 238
>> Total: psf_model_dr10_s.fits  -  81 / 238




### Second & Repeating PSF download pass

If the first pass does not retrieve all PSFs, the next cell retries with:

- larger `Ntry`,
- longer `timeout`,
- visible error messages.

This is useful for distinguishing between ordinary network failures and more systematic problems such as unavailable files or server-side rate limiting.

In [20]:
results = dhcrawl.multicore_download_cutout(
    DIdat,
    show_progress=2000,
    Ncore=3,
    overwrite=False,     # keep successfully downloaded files
    Ntry=3,
    timeout=40000,
    silent_error=False,  
    silent_timeerror=False,  # show Timeout errors so failures can be diagnosed
    silent=True ## Hide detailed info
)

Start Multicore run
======== Parallel Briefing ========
● Function: partial
● Number of loops: 238
● Number of workers: 3
● Backend: thread
● Using try/except: False


partial [thread]:   0%|          | 0/238 [00:00<?, ?it/s]

Done! | Time: 0.0 s
======== Parallel summary ========
● Succeed: 238 / 238
● Failed: 0 / 238
● Warning: 0 / 238



In [22]:
# Remove error logs before retrying.
dhfc.multicore_remove(DIdat.dir_work_list, fna='err*.dat', Ncore=4)

# Second PSF download pass with longer timeout and more retries.
results = dhcrawl.multicore_download_cutout(
    DIdat,
    show_progress=2000,
    Ncore=3,
    overwrite=False,     # keep successfully downloaded files
    Ntry=3,
    timeout=40000,
    silent_error=True,  
    silent_timeerror=False,  # show Timeout errors so failures can be diagnosed
    silent=True ## Hide detailed info
)

# A second identical pass can sometimes recover files after temporary server failures.
# If this does not change the file count, there is no need to keep repeating it.
results = dhcrawl.multicore_download_cutout(
    DIdat,
    show_progress=2000,
    Ncore=3,
    overwrite=False,
    Ntry=3,
    timeout=40000,
    silent_error=False,    # show errors so failures can be diagnosed
    silent_timeerror=False,  # show Timeout errors so failures can be diagnosed
    silent=True ## Hide detailed info
)

# Re-check PSF file availability.
FG = dhfc.FileGroupSearch(
    DirInfo=DIdat,
    fn_set=['psf_model_dr9_s.fits', 'psf_model_dr9_n.fits', 'psf_model_dr10_s.fits']
)


Remove files:   0%|          | 0/238 [00:00<?, ?it/s]

Done! | Time: 0.0 s
Start Multicore run
======== Parallel Briefing ========
● Function: partial
● Number of loops: 238
● Number of workers: 3
● Backend: thread
● Using try/except: False


partial [thread]:   0%|          | 0/238 [00:00<?, ?it/s]

Done! | Time: 14.2 s
======== Parallel summary ========
● Succeed: 25 / 238
● Failed: 0 / 238
● Warning: 213 / 238

Start Multicore run
======== Parallel Briefing ========
● Function: partial
● Number of loops: 238
● Number of workers: 3
● Backend: thread
● Using try/except: False


partial [thread]:   0%|          | 0/238 [00:00<?, ?it/s]

Done! | Time: 0.0 s
======== Parallel summary ========
● Succeed: 238 / 238
● Failed: 0 / 238
● Warning: 0 / 238

Gal Index:  0 - 237
Total NGal :  238
>> File: psf_model_dr9_s.fits  -  140 / 238
>> Err file: err_psf_model_dr9_s.dat  -  98 / 238
>> Total: psf_model_dr9_s.fits  -  238 / 238


>> File: psf_model_dr9_n.fits  -  85 / 238
>> Err file: err_psf_model_dr9_n.dat  -  153 / 238
>> Total: psf_model_dr9_n.fits  -  238 / 238


>> File: psf_model_dr10_s.fits  -  164 / 238
>> Err file: err_psf_model_dr10_s.dat  -  74 / 238
>> Total: psf_model_dr10_s.fits  -  238 / 238




#### What to check after PSF download

After running the previous cell, check:

- whether any error messages appear,
- whether the expected PSF files exist in `gal_dat/`,
- whether `FG.donelist` suggests that many targets are missing.

Some common error messages are:

- **Error code 500** usually means that the target does not exist in the requested survey region. Check whether the target is missing from all relevant regions, such as the north, south, or different DRs.
- **Timeout errors** may mean that the target exists, but the server did not respond. In this case, rerun the download code.
- **Too much request** may occur if too many download requests are sent in a short period. In this case, rerun the download code with a longer timeout and fewer cores.

A few missing files after the first few passes are normal.

### Checking PSF completeness

Identify targets missing both usable north/south PSF products.

**Interpretation:** an empty output means no target is missing both PSF choices. If indices are returned, those targets should be inspected before continuing.

Check if the target galaxy should have PSF. If not, repeat the downloading codes above.

In [12]:
# Identify targets missing both usable north/south PSF products.
# If this list is empty, all targets have at least one PSF option.
np.where((FG.donelist[1] | FG.donelist[2]) == False)


(array([  7,  15,  27,  46,  63,  68,  73,  75,  95, 138, 148, 188, 195,
        198, 210]),)

#### Retrying downloads after removing error logs

By default, if a target already has an error log (Error code 500), the download code keeps that error record and does not try to download the same file again. This behavior prevents the code from repeatedly requesting files that are genuinely unavailable.

However, in some cases, an error log may be created incorrectly. For example, this can happen if the code stops unexpectedly during the download process or if the server has a temporary problem. Therefore, after the normal download attempts are complete, you may want to remove all error logs and retry the download one final time.

Use the following command to remove the existing error logs. Check if the number of successful data has been changed


In [25]:
# Remove error logs before retrying.
res = dhfc.multicore_remove(DIdat.dir_work_list, fna='err*.dat', Ncore=4)

Remove files:   0%|          | 0/238 [00:00<?, ?it/s]

Done! | Time: 0.0 s


and repeat the download

In [26]:
results = dhcrawl.multicore_download_cutout(
    DIdat,
    show_progress=2000,
    Ncore=2,
    overwrite=False,
    Ntry=3,
    timeout=40000,
    silent_error=False,    # show errors so failures can be diagnosed
    silent_timeerror=False,  # show Timeout errors so failures can be diagnosed
    silent=True ## Hide detailed info
)

# Re-check PSF file availability.
FG = dhfc.FileGroupSearch(
    DirInfo=DIdat,
    fn_set=['psf_model_dr9_s.fits', 'psf_model_dr9_n.fits', 'psf_model_dr10_s.fits']
)


Start Multicore run
======== Parallel Briefing ========
● Function: partial
● Number of loops: 238
● Number of workers: 2
● Backend: thread
● Using try/except: False


partial [thread]:   0%|          | 0/238 [00:00<?, ?it/s]

>>>> Error! code:  500 ./gal_dat/gal0001/psf_model_dr9_n.fits
>>>> Error! code:  500 ./gal_dat/gal0000/psf_model_dr9_s.fits
>>>> Error! code:  500 ./gal_dat/gal0000/psf_model_dr10_s.fits
>>>> Error! code:  500 ./gal_dat/gal0002/psf_model_dr9_n.fits
>>>> Error! code:  500 ./gal_dat/gal0005/psf_model_dr9_n.fits
>>>> Error! code:  500 ./gal_dat/gal0004/psf_model_dr9_n.fits
>>>> Error! code:  500 ./gal_dat/gal0007/psf_model_dr9_s.fits
>>>> Error! code:  500 ./gal_dat/gal0006/psf_model_dr9_n.fits
>>>> Error! code:  500 ./gal_dat/gal0007/psf_model_dr9_n.fits
>>>> Error! code:  500 ./gal_dat/gal0010/psf_model_dr9_n.fits
>>>> Error! code:  500 ./gal_dat/gal0011/psf_model_dr9_n.fits
>>>> Error! code:  500 ./gal_dat/gal0007/psf_model_dr10_s.fits
>>>> Error! code:  500 ./gal_dat/gal0012/psf_model_dr9_s.fits
>>>> Error! code:  500 ./gal_dat/gal0013/psf_model_dr9_s.fits
>>>> Error! code:  500 ./gal_dat/gal0013/psf_model_dr10_s.fits
>>>> Error! code:  500 ./gal_dat/gal0012/psf_model_dr10_s.fits
>>>>

## Extract band-specific PSF files

Some downloaded PSF products are multi-extension or multi-band FITS files. MC2fit/GALFIT is easier to run when each band has a standardized PSF filename.

This step extracts the needed PSF images and saves them with band-specific names expected by later steps.

In [27]:
## Extract individual g/r-band PSF FITS files from the downloaded PSF containers.
## In DH_Galfit_Sequence.py, multicore_extract_psf_fits() splits multi-band PSF files
## into separate files such as "...g.fits" and "...r.fits".
res = dhmc2fit.multicore_extract_psf_fits(DIdat.dir_work_list, fna='psf_model_dr9_n.fits', use_try=True)
res = dhmc2fit.multicore_extract_psf_fits(DIdat.dir_work_list, fna='psf_model_dr9_s.fits', use_try=True)
res = dhmc2fit.multicore_extract_psf_fits(DIdat.dir_work_list, fna='psf_model_dr10_s.fits', use_try=True)


======== Parallel Briefing ========
● Function: partial
● Number of loops: 238
● Number of workers: 10
● Backend: thread
● Using try/except: True


Extract PSF FITS:   0%|          | 0/238 [00:00<?, ?it/s]

Done! | Time: 0.5 s
======== Parallel summary ========
● Succeed: 76 / 238
● Failed: 0 / 238
● Warning: 162 / 238

======== Parallel Briefing ========
● Function: partial
● Number of loops: 238
● Number of workers: 10
● Backend: thread
● Using try/except: True


Extract PSF FITS:   0%|          | 0/238 [00:00<?, ?it/s]

Done! | Time: 0.7 s
======== Parallel summary ========
● Succeed: 137 / 238
● Failed: 0 / 238
● Warning: 101 / 238

======== Parallel Briefing ========
● Function: partial
● Number of loops: 238
● Number of workers: 10
● Backend: thread
● Using try/except: True


Extract PSF FITS:   0%|          | 0/238 [00:00<?, ?it/s]


Loop 80: OSError: No SIMPLE card found, this file does not appear to be a valid FITS file. If this is really a FITS file, try with ignore_missing_simple=True
Traceback (most recent call last):
  File "/home/donghyeon/ua/core_code/DH_multicore.py", line 117, in _multicore_worker
    return i, loop_ftn(i), None
              ~~~~~~~~^^^
  File "/home/donghyeon/ua/core_code/DH_MC2fit.py", line 5399, in _extract_psf_fits_one
    primary_header = fits.getheader(
        fn,
        ignore_missing_simple=ignore_missing_simple,
    )
  File "/home/donghyeon/anaconda3/envs/nb7-py313/lib/python3.13/site-packages/astropy/io/fits/convenience.py", line 121, in getheader
    hdulist, extidx = _getext(filename, mode, *args, **kwargs)
                      ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/donghyeon/anaconda3/envs/nb7-py313/lib/python3.13/site-packages/astropy/io/fits/convenience.py", line 1126, in _getext
    hdulist = fitsopen(filename, mode=mode, **kwargs)
  File "/home/dongh

## Download main image products

This section downloads the science images and related products. Depending on the helper function settings, it may prepare:

- image cutouts,
- sigma or inverse-variance-related products,
- maskbits,
- PSF products centered on the target.

These are still raw downloaded files. Later, the notebook chooses the best north/south version and creates standardized links into the fitting directory.

### Download settings for main images

Important parameters in the next cell:

- `region='s'`: request the southern Legacy Survey region first,
- `dr='dr10'`: use Data Release 10 products,
- `psf_mode='target'`: prepare target-centered PSF products,
- `image_size=image_size`: use the cutout size defined earlier,
- `sleep=3`: pause between requests to reduce server load.

If some targets fail, the retry cells below remove temporary error logs and repeat the request.

### Main image download pass

After this cell runs, inspect the output messages. Do not assume all targets succeeded. Missing files are common when many cutouts are requested.

#### What this download cell does

This is the main data-acquisition step. It contacts the Legacy Survey service for each target coordinate and downloads the image products needed for fitting.

Because this step depends on external servers, rerunning may produce different success/failure patterns if the network or server load changes.

#### Retrying downloads after removing error logs

You should run the download cell multiple times. A single run may not download all files, especially when the server is slow, temporarily unavailable, or when many targets are being requested. After each run, check how many files are still missing and rerun the download cell until the number of missing files no longer decreases.



In [32]:
dhfc.multicore_remove(DIdat.dir_work_list, fna="err*.dat", Ncore=Ncore)

## Download the main image products from the Legacy Survey.

dhmc2fit.multicore_crawling(
    DirInfo=DIdat,
    region='s',
    dr='dr10',
    retry=1,
    timeout=5,
    Ncore=4,
    show_progress=-1,
    psf_mode='target',   # also creates target-centered PSF products
    image_size=image_size,
    add_offset = est_sky
)

dhmc2fit.multicore_crawling(
    DirInfo=DIdat,
    region='n',
    dr='dr9',
    retry=1,
    timeout=5,
    Ncore=4,
    show_progress=-1,
    psf_mode='target',   # also creates target-centered PSF products
    image_size=image_size,
    add_offset = est_sky
)


Remove files:   0%|          | 0/238 [00:00<?, ?it/s]

Done! | Time: 0.0 s
======== Parallel Briefing ========
● Function: partial
● Number of loops: 238
● Number of workers: 4
● Backend: thread
● Using try/except: True


Legacy crawling: dr10 s:   0%|          | 0/238 [00:00<?, ?it/s]

Done! | Time: 23.9 s
======== Parallel summary ========
● Succeed: 238 / 238
● Failed: 0 / 238
● Warning: 0 / 238

Gal Index:  0 - 237
Total NGal :  238
>> File: image_dr10_sg.fits  -  163 / 238
>> Err file: err_image_dr10_sg.dat  -  72 / 238
>> Total: image_dr10_sg.fits  -  235 / 238


>> File: sigma_dr10_sg.fits  -  161 / 238
>> Err file: err_sigma_dr10_sg.dat  -  0 / 238
>> Total: sigma_dr10_sg.fits  -  161 / 238


Gal Index:  0 - 237
Total NGal :  238
>> File: image_dr10_sr.fits  -  163 / 238
>> Err file: err_image_dr10_sr.dat  -  70 / 238
>> Total: image_dr10_sr.fits  -  233 / 238


>> File: sigma_dr10_sr.fits  -  154 / 238
>> Err file: err_sigma_dr10_sr.dat  -  0 / 238
>> Total: sigma_dr10_sr.fits  -  154 / 238


======== Parallel Briefing ========
● Function: partial
● Number of loops: 238
● Number of workers: 4
● Backend: thread
● Using try/except: True


Legacy crawling: dr9 n:   0%|          | 0/238 [00:00<?, ?it/s]

Done! | Time: 19.8 s
======== Parallel summary ========
● Succeed: 238 / 238
● Failed: 0 / 238
● Warning: 0 / 238

Gal Index:  0 - 237
Total NGal :  238
>> File: image_dr9_ng.fits  -  85 / 238
>> Err file: err_image_dr9_ng.dat  -  147 / 238
>> Total: image_dr9_ng.fits  -  232 / 238


>> File: sigma_dr9_ng.fits  -  79 / 238
>> Err file: err_sigma_dr9_ng.dat  -  0 / 238
>> Total: sigma_dr9_ng.fits  -  79 / 238


Gal Index:  0 - 237
Total NGal :  238
>> File: image_dr9_nr.fits  -  85 / 238
>> Err file: err_image_dr9_nr.dat  -  147 / 238
>> Total: image_dr9_nr.fits  -  232 / 238


>> File: sigma_dr9_nr.fits  -  79 / 238
>> Err file: err_sigma_dr9_nr.dat  -  0 / 238
>> Total: sigma_dr9_nr.fits  -  79 / 238




array([<DH_FileCheck.FileGroupSearch object at 0x7619ab787450>,
      dtype=object)

## Select good survey data

Legacy Survey products can exist in both northern and southern survey reductions. This notebook checks which version is better for each target and creates standardized file links for downstream fitting.


### Bad-pixel based survey selection

`generate_checkpixel()` counts invalid or problematic pixels for each target and survey region. `CheckPixel` then uses this information to decide which survey product should be used.

In practice:

- use the survey product with fewer bad pixels,
- reject targets where both options are unacceptable,
- keep the selection rule explicit so the analysis is reproducible.

In [33]:
# Build quality-check tables that count bad/empty pixels in downloaded images.
# The north and south products are checked separately because their coverage/quality can differ.
dhfc.generate_checkpixel(DIdat, Ncore=15, fna_image='image_dr9_', regionlist=['n'], silent=True)
dhfc.generate_checkpixel(DIdat, Ncore=15, fna_image='image_dr10_', regionlist=['s'], silent=True)

# Load the check-pixel results.
# bad_pixel_crit is set very high here, so the object mainly summarizes file quality
# and does not aggressively reject targets unless the data are clearly unusable.
CheckPixels = dhfc.CheckPixel(DIdat, bad_pixel_crit=1e9)

# Print indices of targets flagged as bad.
print("Bad data:", np.where(CheckPixels.list_bad))


Check pixels: ng:   0%|          | 0/238 [00:00<?, ?it/s]

Done! | Time: 0.1 s


Check pixels: nr:   0%|          | 0/238 [00:00<?, ?it/s]

Done! | Time: 0.1 s


Check pixels: sg:   0%|          | 0/238 [00:00<?, ?it/s]

Done! | Time: 0.1 s


Check pixels: sr:   0%|          | 0/238 [00:00<?, ?it/s]

Done! | Time: 0.1 s
Region - s Band - g  exist : 161
Region - s Band - r  exist : 154
Region - s Total exist : 154
Region - n Band - g  exist : 79
Region - n Band - r  exist : 79
Region - n Total exist : 76
List South : 154
List North : 61
List Bad : 23
Bad data: (array([  7,  15,  27,  46,  63,  68,  73,  75,  80,  88,  95, 121, 138,
       148, 162, 170, 171, 188, 195, 198, 210, 218, 219]),)


# Manual Files

In [5]:
image_g_list = ['test1.fits', 'test2.fits', 'test3.fits']
psf_g_list = ['test1.fits', 'test2.fits', 'test3.fits']
sigma_g_list = ['test1.fits', 'test2.fits', 'test3.fits']

In [6]:
# DI manages the standardized fitting/output directory tree.
# Each target gets a folder named with folder_rule, e.g. gal0000/, gal0001/, ...
DIdat = dhfc.DirInfo(
    './',
    dir_work_base='gal_dat/',          # downloaded files
    folder_rule="gal%04d/",            # four-digit folder names; supports up to 9999 targets. To use five-digit names, use "gal%05d/"
    coord_array=np.full((len(image_g_list), 2), np.nan),
    idlist=np.arange(len(image_g_list)), # internal target IDs
    # catalogname_list=target_catalog['name'],    # human-readable target names
)

# Print the number of target directories that will be created (= number of targets)
print(len(DIdat.dir_work_list))

3


In [ ]:
dhmc2fit.copy_files_to_gal_folders(image_g_list, output_dir = DIdat.dir_work_base, 
                                   output_filename='image_lg.fits',
                                   mode='symlink' ## copy or symlink
                                  )
dhmc2fit.copy_files_to_gal_folders(psf_g_list, output_dir = DIdat.dir_work_base, 
                                   output_filename='psf_target_lg.fits',
                                   mode='symlink' ## copy or symlink
                                  )
dhmc2fit.copy_files_to_gal_folders(sigma_g_list, output_dir = DIdat.dir_work_base, 
                                   output_filename='sigma_lg.fits',
                                   mode='symlink' ## copy or symlink
                                  )